<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/08-window-functions/03-running-totals-sessionization.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 8.3 — Running totals, moving averages, sessionization

Three production time-series patterns from the same parts: frames
(8.0) + lag (8.2). orders.csv for the money patterns; a small
literal clickstream (with real idle gaps) for sessionization.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("8.3-running-totals-sessionization")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

orders = (
    spark.read.csv(f"{DATA_DIR}/orders.csv", header=True, inferSchema=True)
    .withColumn("revenue", F.round(col("quantity") * col("unit_price"), 2))
)

## Running totals — the default frame, on purpose

ROWS + unique orderBy (date, then order_id): identical to RANGE here
but rerun-stable and cheaper (no peer scan). Cumulative max rides along.

In [ ]:
w_cum = (Window.partitionBy("customer_id")
         .orderBy("order_date", "order_id")
         .rowsBetween(Window.unboundedPreceding, Window.currentRow))

(orders
 .withColumn("order_no", F.count("*").over(w_cum))
 .withColumn("lifetime_spend", F.round(F.sum("revenue").over(w_cum), 2))
 .withColumn("biggest_so_far", F.max("revenue").over(w_cum))
 .filter(col("customer_id").isin("c001", "c003"))
 .select("customer_id", "order_no", "order_date", "revenue",
         "lifetime_spend", "biggest_so_far")
 .orderBy("customer_id", "order_no")
 .show())

## Moving average — "last 7 rows" vs "last 7 days"

Aggregate to the metric's grain FIRST (one row per day), then window.
ROWS counts observations; RANGE on epoch seconds counts calendar time
and shrinks correctly across gap days.

In [ ]:
daily = (orders.groupBy("order_date")
         .agg(F.round(F.sum("revenue"), 2).alias("day_revenue")))

w_rows7 = Window.orderBy("order_date").rowsBetween(-6, Window.currentRow)
w_days7 = (Window.orderBy(col("order_date").cast("timestamp").cast("long"))
           .rangeBetween(-6 * 86400, Window.currentRow))

ma = (daily
      .withColumn("ma7_rows", F.round(F.avg("day_revenue").over(w_rows7), 2))
      .withColumn("ma7_days", F.round(F.avg("day_revenue").over(w_days7), 2))
      .withColumn("days_in_frame", F.count("*").over(w_days7)))
ma.orderBy("order_date").show(20)
# orders.csv has an order every day, so the two agree here — exercise 1
# knocks days out and watches them diverge. (Global windows: fine on 20
# rows, WindowExec warning and one-task sort at scale — 8.0.)

## Month-to-date: partition by the period, cumulate within it

In [ ]:
w_mtd = (Window.partitionBy(F.date_trunc("month", "order_date"))
         .orderBy("order_date", "order_id")
         .rowsBetween(Window.unboundedPreceding, Window.currentRow))

(orders
 .withColumn("mtd_revenue", F.round(F.sum("revenue").over(w_mtd), 2))
 .select("order_id", "order_date", "revenue", "mtd_revenue")
 .orderBy("order_date", "order_id")
 .show(8))

## Sessionization — the flag-and-sum idiom

Clickstream with real idle gaps (events.jsonl spans one minute — too
dense to sessionize meaningfully, so we build the teaching stream).
Three moves: lag -> boundary flag -> running sum of flags.

In [ ]:
events = spark.createDataFrame(
    [
        ("u1", "2026-06-01 08:00:00", "view"),
        ("u1", "2026-06-01 08:03:10", "cart"),
        ("u1", "2026-06-01 08:04:00", "purchase"),
        ("u1", "2026-06-01 09:30:00", "view"),      # 86 min idle -> new session
        ("u1", "2026-06-01 09:31:00", "view"),
        ("u2", "2026-06-01 08:10:00", "view"),
        ("u2", "2026-06-01 08:55:00", "view"),      # 45 min idle -> new session
        ("u2", "2026-06-01 08:58:00", "cart"),
        ("u2", "2026-06-01 09:20:00", "purchase"),  # 22 min -> SAME session
        ("u3", "2026-06-01 23:50:00", "view"),
        ("u3", "2026-06-02 00:10:00", "purchase"),  # crosses midnight, 20 min -> same
    ],
    "user_id STRING, ts_str STRING, action STRING",
).withColumn("ts", F.to_timestamp("ts_str")).drop("ts_str")

GAP_MIN = 30
w_user = Window.partitionBy("user_id").orderBy("ts")
w_user_cum = w_user.rowsBetween(Window.unboundedPreceding, Window.currentRow)

sessions = (
    events
    .withColumn("gap_min",
        (col("ts").cast("long") - F.lag(col("ts").cast("long")).over(w_user)) / 60)
    .withColumn("new_session",
        F.when(col("gap_min").isNull() | (col("gap_min") >= GAP_MIN), 1).otherwise(0))
    .withColumn("session_no", F.sum("new_session").over(w_user_cum))
    .withColumn("session_id",
        F.concat_ws("-", "user_id", col("session_no").cast("string")))
)
sessions.orderBy("user_id", "ts").show(truncate=False)

## Sessions built — analytics is now a plain groupBy

In [ ]:
(sessions.groupBy("session_id")
 .agg(
     F.count("*").alias("events"),
     F.round((F.max(col("ts").cast("long")) - F.min(col("ts").cast("long"))) / 60, 1)
         .alias("duration_min"),
     F.max((col("action") == "purchase").cast("int")).alias("converted"),
 )
 .orderBy("session_id")
 .show())
# u3's midnight-crossing session is ONE session — that's what gap-based
# means. If the business wants calendar-day cuts too, that's another
# flag OR'd into new_session, not a different algorithm.

## Exercises

1. Delete every weekend row from `daily` (dayofweek in (1,7)), rerun
   the two moving averages, and find a date where ma7_rows uses stale
   pre-gap days while ma7_days correctly shrank. `days_in_frame`
   makes it visible.
2. Streaks with the same idiom: on `daily`, flag day-over-day revenue
   DROPS as boundaries and label "growth streaks"; report the longest
   streak's start, end, and total revenue.
3. Add a second session cut: sessions also break at 60 minutes of
   TOTAL duration, even without an idle gap. (Harder than it looks —
   the naive running-duration check re-anchors wrongly; it needs an
   iterative flag or a different anchor. A correct-but-approximate
   version is acceptable; say where it's approximate.)
4. Rewrite sessionization in one spark.sql() query (LAG + SUM OVER,
   nested subqueries or CTEs). Compare the two explain() outputs —
   same Window nodes?
5. events.jsonl bonus: sessionize it with GAP = 20 SECONDS (its
   events are seconds apart) — schema from 6.x, user.id as the key.
   How many sessions does u101 get?

In [ ]:
# your exercise code here